# Task 13

Автоматическое распознавание автомобильного номера на `image.jpg` с использованием предобученной модели `emnist_symbols.h5`.

In [7]:
# Если зависимости не установлены, раскомментируйте и выполните:
# !pip install tf-keras opencv-python
# После установки перезапустите kernel и снова запустите все ячейки.

In [8]:
from pathlib import Path
from urllib.request import urlretrieve

import cv2
import numpy as np
import tf_keras as keras

MODEL_URL = "https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/Computer_Vision/Task_7/emnist_symbols.h5"

# Работает и когда ноутбук запущен из корня репозитория, и из task-13.
cwd = Path.cwd()
base_candidates = [cwd, cwd / "task-13"]
BASE_DIR = next((p for p in base_candidates if (p / "image.jpg").exists()), cwd)

IMAGE_PATH = BASE_DIR / "image.jpg"
MODEL_PATH = BASE_DIR / "emnist_symbols.h5"

if not IMAGE_PATH.exists():
    raise FileNotFoundError(f"Не найдено изображение: {IMAGE_PATH}")

if not MODEL_PATH.exists():
    print("Скачиваю модель...")
    urlretrieve(MODEL_URL, MODEL_PATH)

emnist_labels = [
    48, 49, 50, 51, 52, 53, 54, 55, 56, 57,
    65, 66, 67, 68, 69, 70, 71, 72, 73, 74,
    75, 76, 77, 78, 79, 80, 81, 82, 83, 84,
    85, 86, 87, 88, 89, 90, 97, 98, 99, 100,
    101, 102, 103, 104, 105, 106, 107, 108, 109, 110,
    111, 112, 113, 114, 115, 116, 117, 118, 119, 120,
    121, 122,
]


def emnist_predict_img(model, img):
    img_arr = np.expand_dims(img, axis=0)
    img_arr = 1 - img_arr / 255.0
    img_arr[0] = np.rot90(img_arr[0], 3)
    img_arr[0] = np.fliplr(img_arr[0])
    img_arr = img_arr.reshape((1, 28, 28, 1))
    result = np.argmax(model.predict(img_arr, verbose=0), axis=1)
    return chr(emnist_labels[result[0]])


def img_to_str(model, letters):
    result = ""
    for i in range(len(letters)):
        dn = letters[i + 1][0] - letters[i][0] - letters[i][1] if i < len(letters) - 1 else 0
        result += emnist_predict_img(model, letters[i][2])
        if dn > letters[i][1] / 4:
            result += " "
    return result


img = cv2.imread(str(IMAGE_PATH))
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

thresh = cv2.adaptiveThreshold(
    gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
)

contours, _ = cv2.findContours(thresh, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

largest_rectangle = [0, None]
for cnt in contours:
    approx = cv2.approxPolyDP(cnt, 0.01 * cv2.arcLength(cnt, True), True)
    if len(approx) == 4:
        area = cv2.contourArea(cnt)
        if area > largest_rectangle[0]:
            largest_rectangle = [area, cnt]

if largest_rectangle[1] is None:
    raise RuntimeError("Не удалось найти номерной прямоугольник")

x, y, w, h = cv2.boundingRect(largest_rectangle[1])
vehicle_num = img[y:y + h, x:x + w]

gray_num = cv2.cvtColor(vehicle_num, cv2.COLOR_BGR2GRAY)
_, thresh_num = cv2.threshold(gray_num, 75, 255, cv2.THRESH_BINARY)
img_erode = cv2.erode(thresh_num, np.ones((3, 3), np.uint8), iterations=1)

contours, hierarchy = cv2.findContours(img_erode, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)

letters = []
for idx, contour in enumerate(contours):
    bx, by, bw, bh = cv2.boundingRect(contour)
    if bw > 10 and bh > 10 and hierarchy[0][idx][3] == 0:
        letter_crop = gray_num[by:by + bh, bx:bx + bw]
        size_max = max(bw, bh)
        letter_square = 255 * np.ones((size_max, size_max), dtype=np.uint8)

        if bw > bh:
            y_pos = size_max // 2 - bh // 2
            letter_square[y_pos:y_pos + bh, 0:bw] = letter_crop
        elif bw < bh:
            x_pos = size_max // 2 - bw // 2
            letter_square[0:bh, x_pos:x_pos + bw] = letter_crop
        else:
            letter_square = letter_crop

        letter_resized = cv2.resize(letter_square, (28, 28), interpolation=cv2.INTER_AREA)
        letters.append((bx, bw, letter_resized))

letters.sort(key=lambda item: item[0])

model = keras.models.load_model(str(MODEL_PATH), compile=False)
raw_result = img_to_str(model, letters)
answer = raw_result.replace(" ", "").lower()

print("raw:", raw_result)
print("answer:", answer)

raw: 00 q88
answer: 00q88
